In [9]:
import numpy as np
import pandas as pd
import os

In [10]:
from Age_BMI_loading import age_matrix_vec,bmi_matrix_ls,age_dict

In [11]:
from prevalence import get_overall_prevalence, get_overall_pre_prevalence
from impute_albu import schedule_pre_albuminuria_probability, schedule_albuminuria_probability


In [12]:
N = 8


In [13]:
# get 1990 
albu_status_dict = {}
albu_mat_list_overall = []
A_range = [0.001, 0.005]

for A in A_range:
    albu_status_dict = {}
    for group in range(N):
        i = group
        albu_status_dict[group] = {}
        albu_status = np.zeros((age_matrix_vec[i].shape[0], age_matrix_vec[i].shape[1]))  
        for age in age_dict[group]:
            # bmi_samples = bmi_samples_dict[group][age]  # shape: (n_samples, n_ages)
            #n_samples, n_ages = bmi_samples.shape
            # Create an age matrix with shape (n_samples, n_ages)
            n_samples = age_dict[group][age]
            age_mat = np.tile(np.arange(18, age + 1), (n_samples, 1))
            # idx = group

            # Compute prediabetes probability matrix
            prediabetes_prob_matrix = schedule_pre_albuminuria_probability(age_mat,i,A)
            rand_matrix = np.random.rand(*age_mat.shape)
            prediabetes_matrix = (rand_matrix < prediabetes_prob_matrix) * 0.5
            prediabetes_matrix = np.maximum.accumulate(prediabetes_matrix, axis=1)

            # Compute albu probability matrix
            albu_prob_matrix = schedule_albuminuria_probability(age_mat,i)
            rand_matrix = np.random.rand(*age_mat.shape)
            albu_matrix = np.where(rand_matrix < albu_prob_matrix, 1, 0.5)

            # Combine: take max along each row (for each sample)
            combined_matrix = np.where(prediabetes_matrix == 0.5, albu_matrix, prediabetes_matrix)
            combined_matrix = np.maximum.accumulate(combined_matrix, axis=1)

            albu_status_dict[group][age] = np.max(combined_matrix, axis=1)

    
    albu_mat_list = []
    pre_albu_mat_list = []
    for i in range(8):
        albu_status = np.zeros((age_matrix_vec[i].shape[0], age_matrix_vec[i].shape[1] ))  
        for age, status_vec in albu_status_dict[i].items():
            # Find positions where the first column of age_matrix_vec[i] equals 'age'
            positions = np.where(age_matrix_vec[i][:, 0] == age)[0]
            # Assign the albu status vector for this age group
            albu_status[positions, 0] = status_vec

        n_cols = age_matrix_vec[i].shape[1]
        albu_status_before = np.repeat(albu_status[:, 0][:, np.newaxis], n_cols, axis=1)
        # Process albu status
        # bmi_mat = bmi_matrix_ls[i]
        age_mat = age_matrix_vec[i]

        pre_albu_prob_matrix = schedule_pre_albuminuria_probability(age_mat, i, A)
        rand_matrix = np.random.rand(*age_mat.shape)
        pre_albu_matrix = (rand_matrix < pre_albu_prob_matrix) * 0.5
        
        pre_albu_matrix = np.where(albu_status_before == 0, pre_albu_matrix, albu_status_before)
        pre_albu_matrix = np.maximum.accumulate(pre_albu_matrix, axis=1)
        pre_albu_mat_list.append(pre_albu_matrix)
        
        albu_prob_matrix = schedule_albuminuria_probability(age_mat, i)
        rand_matrix = np.random.rand(*age_mat.shape)
        albu_matrix = np.where(rand_matrix < albu_prob_matrix, 1, 0.5)
        
        combined_matrix = np.where(pre_albu_matrix == 0.5, albu_matrix, pre_albu_matrix)

        albu_status[:, 1:] = combined_matrix[:, 1:]
        albu_status = np.maximum.accumulate(albu_status, axis=1)
        albu_mat_list.append(albu_status)

    overall_prevalence = get_overall_prevalence(age_matrix_vec,albu_mat_list,-2)
    overall_pre_prevalence = get_overall_pre_prevalence(age_matrix_vec,albu_mat_list,-2)

    print(f"A = {A}: Overall prevalence = {overall_prevalence:.4f}, Overall pre-prevalence = {overall_pre_prevalence:.4f}")
    
    # Store the albu_mat_list for this A value
    albu_mat_list_overall.append(albu_mat_list)




A = 0.001: Overall prevalence = 0.0089, Overall pre-prevalence = 0.0593
A = 0.005: Overall prevalence = 0.0334, Overall pre-prevalence = 0.1710


In [14]:
from prevalence import get_prevalence

In [8]:
get_prevalence(age_matrix_vec,albu_mat_list,-2)

array([0.02943916, 0.02609795, 0.0735758 , 0.05085375, 0.04948695,
       0.02890989, 0.03430244, 0.02364404, 0.01848155, 0.02722557,
       0.05137719, 0.07534389])

In [ ]:
# 1.6; 11.1

In [ ]:
# Load albuminuria matrices for different A values
import os

# Load albu_mat_list_overall for different A values
albu_dir = '../data/albu_matrix/'

A_range = [0.001, 0.005]
N = 8

# Create albu_1_mat_storage: list of arrays where each array has shape (len(A_range), albu_matrix.shape[0], albu_matrix.shape[1])
albu_1_mat_storage = []

# For each group i, create an array that stacks matrices from all A values
for i in range(N):
    # Get the shape from the first A value's matrix for group i
    first_matrix = albu_mat_list_overall[0][i]  # A_range[0], group i
    matrix_shape = first_matrix.shape
    
    # Initialize array to hold matrices for all A values for this group
    group_array = np.zeros((len(A_range), matrix_shape[0], matrix_shape[1]))
    
    # Fill the array with matrices from each A value
    for A_idx in range(len(A_range)):
        group_array[A_idx] = albu_mat_list_overall[A_idx][i]
    
    albu_1_mat_storage.append(group_array)


In [ ]:
# Save all matrices in albu_1_mat_storage to npy files
# for i, group_array in enumerate(albu_1_mat_storage):
#     filename = f'../data/albu_matrix/albu_1_mat_group_{i}.npy'
#     np.save(filename, group_array)
#     print(f"Saved group {i} array to {filename}")

# # Load the saved matrices and check their sizes
# loaded_albu_1_mat_storage = []
# for i in range(len(albu_1_mat_storage)):
#     filename = f'../data/albu_matrix/albu_1_mat_group_{i}.npy'
#     loaded_array = np.load(filename)
#     loaded_albu_1_mat_storage.append(loaded_array)
#     print(f"Loaded group {i} array from {filename}, shape: {loaded_array.shape}")

# # Verify the loaded data matches the original
# print(f"\nVerification:")
# print(f"Original albu_1_mat_storage has {len(albu_1_mat_storage)} groups")
# print(f"Loaded albu_1_mat_storage has {len(loaded_albu_1_mat_storage)} groups")

# for i in range(len(albu_1_mat_storage)):
#     original_shape = albu_1_mat_storage[i].shape
#     loaded_shape = loaded_albu_1_mat_storage[i].shape
#     arrays_equal = np.array_equal(albu_1_mat_storage[i], loaded_albu_1_mat_storage[i])
#     print(f"Group {i} - Original shape: {original_shape}, Loaded shape: {loaded_shape}, Arrays equal: {arrays_equal}")

Saved group 0 array to ../data/albu_matrix/albu_1_mat_group_0.npy
Saved group 1 array to ../data/albu_matrix/albu_1_mat_group_1.npy
Saved group 2 array to ../data/albu_matrix/albu_1_mat_group_2.npy
Saved group 3 array to ../data/albu_matrix/albu_1_mat_group_3.npy
Saved group 4 array to ../data/albu_matrix/albu_1_mat_group_4.npy
Saved group 5 array to ../data/albu_matrix/albu_1_mat_group_5.npy
Saved group 6 array to ../data/albu_matrix/albu_1_mat_group_6.npy
Saved group 7 array to ../data/albu_matrix/albu_1_mat_group_7.npy
Loaded group 0 array from ../data/albu_matrix/albu_1_mat_group_0.npy, shape: (2, 88417, 34)
Loaded group 1 array from ../data/albu_matrix/albu_1_mat_group_1.npy, shape: (2, 91217, 34)
Loaded group 2 array from ../data/albu_matrix/albu_1_mat_group_2.npy, shape: (2, 16696, 34)
Loaded group 3 array from ../data/albu_matrix/albu_1_mat_group_3.npy, shape: (2, 16310, 34)
Loaded group 4 array from ../data/albu_matrix/albu_1_mat_group_4.npy, shape: (2, 12057, 34)
Loaded group

In [276]:
print(f"Created albu_1_mat_storage with {len(albu_1_mat_storage)} groups")
print(f"Each group array has shape: {albu_1_mat_storage[0].shape}")

Created albu_1_mat_storage with 8 groups
Each group array has shape: (2, 88417, 34)
